In [118]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [119]:
df = pd.read_csv("../data/mesures_capteurs.csv")
print(df)

    id_mesure           date_heure id_capteur batiment  temperature  humidite  \
0       M0413  2026-01-22 04:00:00       C005     B002        25.46     58.06   
1       M0290  2026-01-17 01:00:00       C002     B001        24.00     79.73   
2       M0077  2026-01-08 04:00:00       C005     B002        25.82     54.47   
3       M0079  2026-01-08 06:00:00       C007     B003        28.23     69.39   
4       M0183  2026-01-12 14:00:00       C003     B001        20.58     53.80   
..        ...                  ...        ...      ...          ...       ...   
600     M0072  2026-01-07 23:00:00       C012     B004        23.67     60.28   
601     M0107  2026-01-09 10:00:00       C011     B004        24.34     61.42   
602     M0271  2026-01-16 06:00:00       C007     B003        23.73     63.49   
603     M0436  2026-01-23 03:00:00       C004     B002        24.83     79.77   
604     M0103  2026-01-09 06:00:00       C007     B003        17.02     52.04   

     pression  consommation

### Partie 1 – Gestion des doublons 

##### 1) vérifier l’existence de doublons dans df

In [120]:
df[df.duplicated()]

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
183,M0599,2026-01-29 22:00:00,C011,B004,22.02,68.09,1005.90,227.81,OK
231,M0026,2026-01-06 01:00:00,C002,B001,22.87,77.99,1010.15,213.19,OK
355,M0147,2026-01-11 02:00:00,C003,B001,26.14,84.97,1003.59,142.31,OK
538,M0456,2026-01-23 23:00:00,C012,B004,20.68,72.69,1022.77,309.01,OK
539,M0302,2026-01-17 13:00:00,C002,B001,22.05,58.26,1007.30,140.42,OK


##### 2) le cas échéant, supprimer les doublons puis vérifier la suppression 

In [126]:
df = df.drop_duplicates()

In [127]:
df[df.duplicated()]


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat


In [130]:
df = df.dropna(subset=["etat"]).reset_index(drop="true")

### Partie 2 – Sélection de y (cible) et X (caractéristiques) 

In [133]:
y = df["etat"]
X = df[["temperature", "humidite", "pression", "consommation"]]

In [131]:
print("Les 5 premières lignes de X :")
print(X.head())

Les 5 premières lignes de X :
   temperature  humidite  pression  consommation
0        25.46     58.06   1008.95        287.28
1        24.00     79.73    993.39        116.20
2        25.82     54.47   1010.32        288.50
3        28.23     69.39   1019.62        136.65
4        20.58     53.80   1016.58        182.62


Dans notre contexte , on a un probleme de classification.

In [132]:
print("Les 5 premières lignes de y :")
print(y.head())

Les 5 premières lignes de y :
0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: str


##### 3) Quel est le type du problème de machine learning ? 

le type de probleme de machine learning qu'on a est le probleme de la classification

### Partie 3 – Découpage Train/Test 

In [134]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% des données pour le test
    random_state=42,    # permet de garantir la reproductibilité du découpage
    stratify=y          # permet de conserver les mêmes proportions de classes qu'à l'origine
)

### Partie 4 – Gestion des valeurs manquantes 

##### 1) Vérifier l’existence de valeurs manquantes 

In [136]:
print("les valeurs manquantes dans X_train :")
print(X_train.isnull().sum())

print("\nles valeurs manquantes dans X_test :")
print(X_test.isnull().sum())

les valeurs manquantes dans X_train :
temperature     5
humidite        4
pression        5
consommation    3
dtype: int64

les valeurs manquantes dans X_test :
temperature     1
humidite        1
pression        0
consommation    2
dtype: int64


### 2) Sélectionner SimpleImputer avec la médiane

In [137]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

### 3) Qu’est ce qui justifie le choix de la médiane ? 

la médiane offre une estimation plus robuste et fiable du centre des données lorsque celles-ci peuvent contenir du bruit ou des valeurs extrêmes . ce qui est fréquent avec des données de capteurs IoT.

##### 4) Trouver les paramètres (médianes) de l’imputeur sur X_train

In [138]:
imputer.fit(X_train)

print("Médianes calculées par colonne :")
print(imputer.statistics_)

Médianes calculées par colonne :
[  24.9    65.38 1012.3   206.59]


##### 5) Déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test

In [139]:
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)